# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook shows how to load, examine, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library in a reproducible way, referencing each dataset entity by its Croissant `@id`.

### Dataset Source
The dataset is described by a Croissant schema, accessible at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using the `mlcroissant` package.

*We reference all Croissant entities (record sets, fields, etc.) by their `@id`.*

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display some core metadata
md = dataset.metadata
print(f"Dataset title: {md.name}\n")
print(f"Description: {md.description}\n")
print(f"Published: {getattr(md, 'date_published', 'N/A')}")
print(f"Croissant version: {getattr(md, 'conforms_to', 'N/A')}")

## 2. Data Overview
Let's inspect the available record sets and their fields. All references are by `@id`, in accordance with Croissant best practices.

*RecordSet, Field, and Column `@id`s ensure reproducible access to the correct dataset components.*

In [ ]:
# List all top-level Croissant record sets by @id with their names and contained field ids
print("RecordSets in this dataset:")
for rs in dataset.record_sets:
    print(f"  RecordSet @id: {rs.id}")
    print(f"    name: {rs.name}")
    print(f"    contains fields (by @id):")
    for field in rs.fields:
        print(f"      - {field.id} (name: {field.name}) | type: {field.data_type}")
    print()

## 3. Data Extraction
We will extract data from each available `RecordSet` using the Croissant `@id`. All columns of each record set are referenced by their unique `@id`.

Below, we load the data for each top-level record set into a pandas DataFrame, stored in a `dataframes` dictionary.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    df = pd.DataFrame(list(dataset.records(record_set=record_set_id)))
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records from RecordSet @id: {record_set_id}")
    print(f"  Columns: {df.columns.tolist()}")
    print()
# Choose one main RecordSet for deeper analysis (e.g., the first one)
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Using RecordSet @id for analysis: {selected_record_set_id}")
    df = dataframes[selected_record_set_id]
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Below, we demonstrate filtering and normalization of a chosen numeric field, specified by its Croissant `@id`. 

- *Filtering* records by a numeric threshold.
- *Normalizing* the field.
- *Grouping* by a categorical field (all with references by `@id`).

In [ ]:
# Select appropriate field @id for numeric analysis (e.g., find first Float/Integer field)
selected_field_id = None
group_field_id = None

for rs in dataset.record_sets:
    if rs.id == selected_record_set_id:
        for field in rs.fields:
            if not selected_field_id and field.data_type in ['Float', 'Integer', 'Number']:
                selected_field_id = field.id
            if not group_field_id and field.data_type == 'Text':
                group_field_id = field.id
        break
if selected_field_id is None:
    raise RuntimeError("No numeric field found for EDA")

print(f"Using numeric field @id: {selected_field_id}")
if group_field_id:
    print(f"Using categorical/group field @id: {group_field_id}")

# Remove entries with missing/invalid numeric data
numeric_col = selected_field_id
df = df.copy() # Don't modify the original
df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')
filtered_df = df[df[numeric_col] > 0]

print(f"Filtered to {len(filtered_df)} rows with {numeric_col} > 0")

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
print(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

# If group field available, show group means
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_col].mean().reset_index().sort_values(numeric_col, ascending=False)
    print(f"Mean by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the chosen numeric field and its normalized values. Grouped barplots are included if a categorical field is available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.histplot(filtered_df[numeric_col], bins=30, kde=True)
plt.title(f"Distribution of {numeric_col} (by @id)")
plt.xlabel(numeric_col)
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10, 4))
sns.histplot(filtered_df[f"{numeric_col}_normalized"], bins=30, kde=True)
plt.title(f"Distribution of normalized {numeric_col}")
plt.xlabel(f"{numeric_col}_normalized")
plt.ylabel("Count")
plt.show()

# If a group field exists, show a bar plot of mean values
if group_field_id and group_field_id in filtered_df.columns:
    topn = filtered_df[group_field_id].value_counts().head(10).index.tolist()
    group_means = filtered_df[filtered_df[group_field_id].isin(topn)].groupby(group_field_id)[numeric_col].mean()
    group_means.plot(kind='bar', figsize=(10,4))
    plt.title(f"Mean {numeric_col} per {group_field_id} (top 10)")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_col}")
    plt.show()

## 6. Conclusion
We demonstrated loading and exploring a Croissant-defined dataset end-to-end, using `mlcroissant` to load metadata, list record sets, extract and analyze records, and visualize both numeric fields and grouped summaries.

- All dataset entities (record sets, fields, columns) were referenced exclusively via their Croissant `@id`s for full reproducibility.
- The FAIR^2 dataset contains rich protein mass spectrometry data, and can be processed programmatically using the schema information for flexible, robust workflows.

Next steps could include custom filtering, integrating with other datasets by @id, or preparing the data for machine learning tasks.